# 实验四：Vector RAG / GraphRAG / WikiRAG
使用 `myenv` 内核和真实的 `BAAI/bge-small-zh-v1.5` 嵌入模型，仅在 CPU 上运行。主检索使用固定40条三元组和9个条目；API抽取、生成、社区摘要和条目重写由 `hw4_rag_api.py` 单独完成，不覆盖主评测。密钥只读环境变量，不在Notebook保存或显示。

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import csv, json
from hw4_rag import OUT, QUESTIONS, run_all
print('问题总数:', len(QUESTIONS), '；内置 15 + 自拟 5')
print('输出目录:', OUT)

## 主检索评测
默认复用已有主结果。确需重跑时手动将 REBUILD_BASELINE 改为 True；重跑会覆盖实验四主结果，但不覆盖其他作业。API补充不需要重跑此单元。

In [ ]:
REBUILD_BASELINE = False
if REBUILD_BASELINE or not (OUT / 'metrics_20.csv').exists():
    run_all()
else:
    print('复用原主检索结果，不覆盖已有文件。')

## 报告表 1、表 3、表 4：配置、20 题指标、内置 15 题 K 扫描

In [ ]:
print(json.dumps(json.loads((OUT / 'config.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
for filename in ('metrics_20.csv', 'k_scan_15.csv', 'hybrid_metrics_20.csv'):
    print('\n', filename)
    with (OUT / filename).open(encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f): print(row)

## 图 1~3：Q01 检索排序、实体图、结果曲线

In [ ]:
for filename in ('vector_q01.png', 'graph.png', 'wiki_entries.png', 'recall_20.png', 'k_sensitivity_15.png'):
    print(filename)
    display(Image(filename=str(OUT / filename), width=800))

## 手算 MRR、失败案例和选做 RRF
Q01（多跳）、Q08（事实）、Q14（全局）选自 WikiRAG，首命中位置必须从真实检索记录核对。RRF 是选做实验；是否改进要以实际数据为准。

In [ ]:
print((OUT / 'mrr_manual_3.txt').read_text(encoding='utf-8'))
details = json.loads((OUT / 'retrieval_details.json').read_text(encoding='utf-8'))
for system in ('Vector RAG', 'GraphRAG', 'WikiRAG'):
    failures = [r for r in details if r['system'] == system and r['recall'] < 1]
    print(system, '严格漏检案例:', [(r['id'], r['evidence'], r['hits']) for r in failures])
    if len(failures) < 2: print('严格漏检不足 2 个；报告须注明，不能虚构。')

## 忠实度人工检查说明
现在使用 DeepSeek 的真实返回，3范式 × Q01/Q06/Q11 共9个回答。原Top-5文档不重排，使用其完整原文作上下文。辅助评分与最终人工复核分开；Graph Q11、Wiki Q01/Q11拒答可以忠实但不完整。旧示范回答只保留为历史记录。

## API补充：真实调用与结果索引
运行前在启动Jupyter的终端设置 LLM_API_KEY、LLM_API_BASE、LLM_MODEL，见复现说明。默认只读已经跑好的结果；只有手动设置 RUN_API=True 才发送可能计费的请求。新复跑请更换 API_OUT，避免混淆历史记录。

In [ ]:
import hw4_rag_api as api
API_OUT = OUT / 'api_20261005'
RUN_API = False
if RUN_API:
    api.cached_call(API_OUT / 'api_probe.json', '连通性测试：只回答‘API 可用’。', {'phase': 'probe'})
    api.extraction_phase(API_OUT, 'prompt_iterations', list(api.EXTRACT_PROMPTS), ['doc04', 'doc10', 'doc13'])
    print('先核对三版输出，再运行后面的最终抽取单元。')
if (API_OUT / 'prompt_iterations_results.json').exists():
    for r in json.loads((API_OUT / 'prompt_iterations_results.json').read_text(encoding='utf-8')):
        print(r['version'], r['doc_id'], r['raw_output'])

In [ ]:
FINAL_VERSION = 'V3'
if RUN_API:
    api.extraction_phase(API_OUT, 'full_extract', [FINAL_VERSION], list(api.source_constants()['CORPUS']))
    api.generate_answers(API_OUT)
    api.community_summaries(API_OUT)
    api.wiki_rewrites(API_OUT)
    api.evaluate_supplements(API_OUT)

In [ ]:
for name in ('prompt_selection.json', 'full_extract_summary.json', 'llm_extracted_graph_config.json'):
    if (API_OUT / name).exists():
        print(name, (API_OUT / name).read_text(encoding='utf-8'))
if (API_OUT / 'answers_9.json').exists():
    for r in json.loads((API_OUT / 'answers_9.json').read_text(encoding='utf-8')):
        print(r['system'], r['qid'], r['doc_ids'], r['raw_output'])
if (API_OUT / 'supplement_metrics_20.csv').exists():
    print((API_OUT / 'supplement_metrics_20.csv').read_text(encoding='utf-8-sig'))
print('人工复核材料：', API_OUT / '人工复核材料.md')